# Diagnosing a metric that moved

Core stage, lesson C13. Part of [datafromzero](https://datafromzero.github.io), data and text CC BY 4.0.

**Goals:** walk from "the metric moved" to a cause in a fixed order: rule out the data pipeline, then the calendar, then split by segment, then tell a change of mix from a change inside the segments, and end with a verdict in plain words.

**How to use this notebook:** run the cells from top to bottom. When in doubt, **Run all from the top**: restart the session and run every cell from the first.
Every exercise has a cell for your code and a check cell that compares your answer with a known answer from the app-events data. Hints and solutions are folded away
under each exercise, free to open. The working cell of Exercise 2 starts empty: you write it. The data is synthetic: a script made it from stated rules.

## Setup

In [ ]:
import math
import numbers

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

# Run this cell as it is. It also holds the answer checker used by the exercise cells.
# Run all from the top: restart the session, run every cell from the first, then read the last cell.
EXERCISES = ['Exercise 1', 'Exercise 2', 'Exercise 3']


RESULTS = {}


def same(got, expected, tol=1e-9):
    """True when two values agree: floats within `tol`, lists and dicts item by item."""
    try:
        if isinstance(expected, dict):
            return isinstance(got, dict) and got.keys() == expected.keys() and all(same(got[k], expected[k], tol) for k in expected)
        if isinstance(expected, (list, tuple)):
            return isinstance(got, (list, tuple)) and len(got) == len(expected) and all(same(g, e, tol) for g, e in zip(got, expected))
        if isinstance(expected, bool) or isinstance(got, bool):
            return type(got) is type(expected) and got == expected
        if isinstance(expected, float):
            return isinstance(got, numbers.Real) and math.isclose(got, expected, rel_tol=1e-9, abs_tol=tol)
        if isinstance(expected, int):
            return isinstance(got, numbers.Integral) and got == expected
        return type(got) is type(expected) and got == expected
    except Exception:
        return False


def check(label, name, expected, traps=None, tol=1e-9):
    """Compare the variable called `name` with the known answer. It never stops Run all."""
    got = globals().get(name)
    if got is None:
        RESULTS[label] = None
        print(f"{label}: give {name} a value, then run this cell again.")
        return False
    if same(got, expected, tol):
        RESULTS[label] = True
        print(f"{label}: correct.")
        return True
    RESULTS[label] = False
    print(f"{label}: not yet. Expected {expected!r}, but {name} is {got!r}.")
    for wrong, message in (traps or {}).items():
        if same(got, wrong, tol):
            print("Hint:", message)
    return False


def check_all(label, known):
    """Compare several variables with their known answers: name -> (answer, tolerance). It never stops Run all."""
    got = {name: globals().get(name) for name in known}
    if all(value is None for value in got.values()):
        RESULTS[label] = None
        print(f"{label}: give " + ", ".join(known) + " a value, then run this cell again.")
        return False
    wrong = [name for name, (want, tol) in known.items() if got[name] is None or not same(got[name], want, tol)]
    RESULTS[label] = not wrong
    if wrong:
        print(f"{label}: not yet. Check " + ", ".join(wrong) + ".")
    else:
        print(f"{label}: correct.")
    return not wrong

# One look for every chart of the lesson: small figure, no top and right frame, no legend frame, space made automatically.
BLUE, GREEN, PURPLE, GOLD, RED = "#0072b2", "#00a07a", "#9a6be0", "#ad9100", "#c0392b"  # the colors of the site's figures
plt.rcParams.update({
    "axes.prop_cycle": plt.cycler(color=[BLUE, GREEN, PURPLE, GOLD]),
    "figure.figsize": (4.5, 2.8),
    "figure.constrained_layout.use": True,
    "font.size": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "legend.frameon": False,
})

## Data

In [ ]:
# The app-events tables (version 1), read from the release tag v0.2.0; when the tag cannot be reached, from the copy in a clone of the repository.
BASE = "https://raw.githubusercontent.com/datafromzero/datafromzero.github.io/v0.2.0/datasets/app-events/v1/"
LOCAL = "../../datasets/app-events/v1/"


def load(name, **kwargs):
    """Read one CSV file of app-events: from the release tag, else from the local clone."""
    try:
        return pd.read_csv(BASE + name, **kwargs)
    except Exception:
        return pd.read_csv(LOCAL + name, **kwargs)


users = load("users.csv")
events = load("events.csv", parse_dates=["event_ts", "received_at"])         # 13 MB: a few seconds

# Known answer from the dataset README: 21,841 people and 177,379 events.
assert (len(users), len(events)) == (21841, 177379)

# The platform of a person is the platform of their signup_complete event; its time is the signup time.
signup = events[events["event_name"] == "signup_complete"].drop_duplicates("user_id").set_index("user_id")
users["platform"] = users["user_id"].map(signup["platform"])
users["signup_ts"] = users["user_id"].map(signup["event_ts"])

# Two 0/1 columns, each within 14 days of the signup: bought (a purchase event) and onboarded (an onboarding_complete event).
def within14(name):
    rows = events[events["event_name"] == name].merge(users[["user_id", "signup_ts"]], on="user_id")
    return users["user_id"].isin(rows[rows["event_ts"] <= rows["signup_ts"] + pd.Timedelta(days=14)]["user_id"]).astype(int)


users["bought"] = within14("purchase")
users["onboarded"] = within14("onboarding_complete")

# People who signed up by June 16 have a full 14 days of follow-up (the data ends June 30). Three windows of signups, around a marketing campaign:
full = users[users["signup_date"] <= "2026-06-16"]
base = full[(full["signup_date"] >= "2026-03-09") & (full["signup_date"] <= "2026-04-12")]      # the five weeks before the campaign
camp = full[(full["signup_date"] >= "2026-04-13") & (full["signup_date"] <= "2026-06-07")]      # the campaign weeks
after = full[full["signup_date"] >= "2026-06-08"]                                               # after it stopped, June 8 to 16
assert (len(base), len(camp)) == (3622, 9110)         # the README: 3,622 and 9,110 people
assert round(float(base["bought"].mean()), 3) == 0.129 and round(float(camp["bought"].mean()), 3) == 0.099      # 12.9 and 9.9 percent

print(f"Data loaded: {len(users):,} people and {len(events):,} events; conversion {100 * base['bought'].mean():.2f}% before the campaign, {100 * camp['bought'].mean():.2f}% during it.")

## Unit 1: Is the pipeline telling the truth

**Predict first.** By `view_paywall` alone the views fell by about a third between the four weeks before May 18 and the four weeks from May 18. The cell counts the events whose name contains "paywall", by name, in the same two windows. Did people look at the paywall about a third less often?

In [ ]:
paywall = events[events["event_name"].str.contains("paywall")]        # every event whose name contains "paywall"
windows = {"Apr 20 to May 17": ("2026-04-20", "2026-05-18"), "May 18 to Jun 14": ("2026-05-18", "2026-06-15")}
for label, (start, stop) in windows.items():
    rows = paywall[(paywall["event_ts"] >= start) & (paywall["event_ts"] < stop)]
    print(label, rows["event_name"].value_counts().to_dict(), len(rows))
renamed = paywall[paywall["event_name"] != "view_paywall"]            # the names other than view_paywall
print(renamed["platform"].value_counts().to_dict(), renamed["event_ts"].min())

**Predict first.** On March 17 the collector was down for part of the afternoon. The cell counts the events of that day per hour twice: by when each event happened and by when it was received. Counted by the time of receipt, what will the count do between 14:00 and 19:59, and what will it do at 20:00?

In [ ]:
day = events[(events["event_ts"] >= "2026-03-17") & (events["event_ts"] < "2026-03-18")]
arrived = events[(events["received_at"] >= "2026-03-17") & (events["received_at"] < "2026-03-18")]
per_hour = pd.DataFrame({"happened": day.groupby(day["event_ts"].dt.hour).size(), "received": arrived.groupby(arrived["received_at"].dt.hour).size()}).fillna(0).astype(int)
for hour in range(13, 22):
    print(f"{hour:02d}:00  happened {per_hour.loc[hour, 'happened']:>3}  received {per_hour.loc[hour, 'received']:>3}")
lag_hours = (events["received_at"] - events["event_ts"]).dt.total_seconds() / 3600
outage = (events["event_ts"] >= "2026-03-17 14:00") & (events["event_ts"] < "2026-03-17 20:00")
print(int(outage.sum()), int((lag_hours >= 1).sum()), int(((lag_hours >= 1) & ~outage).sum()), int((lag_hours < 0).sum()))

## Unit 2: Is it the calendar

**Predict first.** Monday, May 25, 2026 was Memorial Day, a US public holiday. The cell counts the return visits (`session_start`) on five Mondays, for people in `north_america` and for people in the other three regions. Will the holiday show in both groups, in one, or in neither?

In [ ]:
visits = events[events["event_name"] == "session_start"].merge(users[["user_id", "region"]], on="user_id")
visits = visits[visits["event_ts"].dt.dayofweek == 0]                        # Mondays only: compare a weekday with the same weekday
visits["day"] = visits["event_ts"].dt.strftime("%Y-%m-%d")
mondays = ["2026-05-11", "2026-05-18", "2026-05-25", "2026-06-01", "2026-06-08"]
mon = visits[visits["day"].isin(mondays)]
na = mon[mon["region"] == "north_america"].groupby("day").size()
other = mon[mon["region"] != "north_america"].groupby("day").size()
for day in mondays:
    print(day, int(na[day]), int(other[day]))
around = [d for d in mondays if d != "2026-05-25"]
print(float(na[around].mean()), float(other[around].mean()))
late_rows = events[(events["received_at"] - events["event_ts"]) >= pd.Timedelta(hours=1)]     # events received an hour or more after they happened
print([int((late_rows["event_ts"].dt.strftime("%Y-%m-%d") == day).sum()) for day in mondays])

## Unit 3: Where did it move

**Predict first.** The 14-day conversion fell from 12.95 to 9.88 percent between the five weeks before the campaign and the campaign. The cell splits both periods by channel, by platform and by region and prints each segment's share of the people and its conversion. In which split will every segment hold almost still while the shares move?

In [ ]:
def split(col):
    """Per segment: its share of the people and its conversion."""
    for seg in sorted(base[col].unique()):
        b, c = base[base[col] == seg], camp[camp[col] == seg]
        print(f"  {seg:<15} share {100 * len(b) / len(base):4.1f}% -> {100 * len(c) / len(camp):4.1f}%   conversion {100 * b['bought'].mean():4.1f}% -> {100 * c['bought'].mean():4.1f}%")


for col in ["channel", "platform", "region"]:
    print(col)
    split(col)

## Unit 4: Mix or rate

**Predict first.** Give the campaign people the conversion rate of each channel from the five weeks before, and keep the channel mix of the campaign. What conversion would the campaign weeks have had: closer to 12.95 or to 9.88 percent?

In [ ]:
w0, w1 = base["channel"].value_counts(normalize=True), camp["channel"].value_counts(normalize=True)      # shares of the people
r0, r1 = base.groupby("channel")["bought"].mean(), camp.groupby("channel")["bought"].mean()             # conversion by channel
n0, n1 = base["channel"].value_counts(), camp["channel"].value_counts()
before, during = base["bought"].mean(), camp["bought"].mean()
counterfactual = (w1 * r0).sum()                                    # the later shares with the earlier rates
mix_part, rate_part = counterfactual - before, during - counterfactual
print(f"{100 * before:.2f} {100 * during:.2f} {100 * (during - before):.2f}")
print(f"{100 * counterfactual:.2f} {100 * mix_part:.2f} {100 * rate_part:.2f}")
se = np.sqrt((w1 ** 2 * (r1 * (1 - r1) / n1 + r0 * (1 - r0) / n0)).sum())      # the standard error of the rate part
print(f"{100 * se:.2f}", round(float(rate_part / se), 2))
# the other order: the earlier shares for the rate part, so the mix part is what is left, at the later rates
rate_early = (w0 * (r1 - r0)).sum()
mix_late = (during - before) - rate_early
interaction = rate_part - rate_early                                # the part that depends on both changes at once
se_early = np.sqrt((w0 ** 2 * (r1 * (1 - r1) / n1 + r0 * (1 - r0) / n0)).sum())
print(f"{100 * mix_late:.2f} {100 * rate_early:.2f} {100 * interaction:+.2f}")
print(f"{100 * se_early:.2f}", round(float(rate_early / se_early), 2))

## Unit 5: Say it, and say what would show you wrong

**Predict first.** The campaign stopped on June 7. If a change of mix explains the drop, what should happen to the paid_social share and to the conversion of the people who signed up from June 8 to 16?

In [ ]:
print(len(after), f"{100 * (after['channel'] == 'paid_social').mean():.1f}%", f"{100 * after['bought'].mean():.1f}%")
verdict = (f"Conversion fell {100 * (before - during):.2f} points, from {100 * before:.2f}% to {100 * during:.2f}%. {100 * -mix_part:.2f} points come from who signed up: "
           f"paid_social went from {100 * w0['paid_social']:.1f}% to {100 * w1['paid_social']:.1f}% of the signups. The other {100 * -rate_part:.2f} points, inside the channels, are within noise (standard error {100 * se:.2f}).")
print(verdict)

## Finish it

### Exercise 1: Finish it

**Task:** After the campaign ended, conversion recovered. Split the change from the campaign weeks to the people who signed up from June 8 to 16 into its mix part and its rate part, by channel, in the order of unit 4: the mix part is the later shares with the earlier rates, minus the earlier blend; the rate part is the later shares times the change of each rate. The shares and rates of both periods are given. Write the two sums, in points.

**Expected:** `answer_1` is a list of two numbers, the mix part and the rate part in percentage points, each rounded to 2 decimals.

In [ ]:
shares_camp = camp["channel"].value_counts(normalize=True)           # given: shares of the campaign weeks
shares_after = after["channel"].value_counts(normalize=True)         # given: shares of June 8 to 16
rate_camp = camp.groupby("channel")["bought"].mean()                 # given: conversion by channel, campaign weeks
rate_after = after.groupby("channel")["bought"].mean()               # given: and June 8 to 16
# the mix part and the rate part of the change in conversion, in points
answer_1 = None  # replace None with [mix, rate]: points, each rounded to 2 decimals

In [ ]:
_ = check("Exercise 1", "answer_1", [2.65, 2.13], traps={(2.13, 2.65): "Those are the two parts in the other order. The mix part comes first.", (3.96, 0.82): "That is the other order of the split: the earlier shares for the rate part. Use the order of unit 4."})

<details>
<summary>Hint</summary>

Unit 4: the mix part is `((w1 - w0) * r0).sum()` there, the later shares with the earlier rates minus the earlier blend, and the rate part is `(w1 * (r1 - r0)).sum()`. Use the variables of this exercise, and multiply by 100 for points.

</details>

<details>
<summary>Solution</summary>

```python
mix_part = ((shares_after - shares_camp) * rate_camp).sum()
rate_part = (shares_after * (rate_after - rate_camp)).sum()
answer_1 = [round(float(100 * mix_part), 2), round(float(100 * rate_part), 2)]
rate_other = (shares_camp * (rate_after - rate_camp)).sum()          # the other order: the earlier shares for the rate part
print(round(float(100 * (mix_part + rate_part - rate_other)), 2), round(float(100 * rate_other), 2))
```

The mix part is +2.65 points and the rate part +2.13, in the order of unit 4. The other order gives 3.96 and 0.82: the interaction is 1.31 points, large here because the shares moved a lot, so the split depends on the order and the total, 4.78, does not. The paid_social share fell from 37.7% to 9.3%, which lifts the blend. The standard error of the rate part is 1.13 points, so 2.13 is 1.88 standard errors: not clearly more than chance.

</details>

## On your own

### Exercise 2: On your own

**Task:** The share of people who finish onboarding within 14 days (an `onboarding_complete` event; the column `onboarded` is 0 or 1) also fell between the five weeks before the campaign and the campaign. Before you start, say in one sentence what the result looks like: four numbers. Use the channel decomposition of unit 4 on `onboarded` and set `answer_2` to the list [change, mix part, rate part, standard error of the rate part], in percentage points, each rounded to 2 decimals. Write the code in the empty cell of the notebook.

**Expected:** `answer_2` is a list of four numbers.

In [ ]:
_ = check("Exercise 2", "answer_2", [-12.02, -8.35, -3.67, 1.19], traps={(-3.07, -2.86, -0.21, 0.6): "These are the numbers of conversion (the column bought), not of onboarding. Use the column onboarded."})

<details>
<summary>Hint</summary>

Unit 4 with the column `onboarded` in place of `bought`: shares, rates and counts by channel for `base` and `camp`, the counterfactual `(w1 * r0).sum()`, the two parts, and the standard error `np.sqrt((w1 ** 2 * (r1 * (1 - r1) / n1 + r0 * (1 - r0) / n0)).sum())`.

</details>

<details>
<summary>Half-solution</summary>

Expected output in words: *"A drop of about a dozen points, a mix part that is most of it, and a rate part with its standard error."*

```python
t0, t1 = base["onboarded"].mean(), camp["onboarded"].mean()
q0, q1 = base.groupby("channel")["onboarded"].mean(), camp.groupby("channel")["onboarded"].mean()
```

</details>

<details>
<summary>Solution</summary>

```python
t0, t1 = base["onboarded"].mean(), camp["onboarded"].mean()
q0, q1 = base.groupby("channel")["onboarded"].mean(), camp.groupby("channel")["onboarded"].mean()
counter = (w1 * q0).sum()
se_o = np.sqrt((w1 ** 2 * (q1 * (1 - q1) / n1 + q0 * (1 - q0) / n0)).sum())
answer_2 = [round(float(100 * (t1 - t0)), 2), round(float(100 * (counter - t0)), 2), round(float(100 * (t1 - counter)), 2), round(float(100 * se_o), 2)]
```

Onboarding fell from 74.2% to 62.2%, 12.02 points. The change of mix explains 8.35 of them. The rate part, -3.67 points, is about 3 standard errors (the standard error is 1.19 points), so something besides the mix lowered onboarding inside the channels; look for a group with zero onboarding.


</details>

## Check yourself

### Exercise 3: Check yourself: spot the flaw

**Task:** Spot the flaw. A colleague reads the change of conversion inside each channel and says that every channel that moved by a point or more got better or worse. The p-value of each channel's change is given. Fix the line that decides which channels moved so that it follows from the p-value, with a bar of 0.05, and set `answer_3` to [number of channels that moved, the smallest p-value rounded to 2 decimals].

**Expected:** `answer_3` is a list of a whole number and a number.

In [ ]:
def p_change(channel):
    """Given: the two-sided p-value of one channel's change (z test of two proportions)."""
    b, c = base[base["channel"] == channel], camp[camp["channel"] == channel]
    x0, x1, m0, m1 = int(b["bought"].sum()), int(c["bought"].sum()), len(b), len(c)
    pooled = (x0 + x1) / (m0 + m1)
    z = (x1 / m1 - x0 / m0) / np.sqrt(pooled * (1 - pooled) * (1 / m0 + 1 / m1))
    return float(2 * stats.norm.sf(abs(z)))


channels = sorted(full["channel"].unique())
p_values = {channel: p_change(channel) for channel in channels}      # given
change = {channel: camp[camp["channel"] == channel]["bought"].mean() - base[base["channel"] == channel]["bought"].mean() for channel in channels}

In [ ]:
# given: channels (the five channels), change (the change of conversion in each) and p_values (the p-value of each change)
moved = [channel for channel in channels if abs(change[channel]) >= 0.01]     # colleague: "moved = changed by a point or more"
answer_3 = [len(moved), round(min(p_values.values()), 2)]

In [ ]:
_ = check("Exercise 3", "answer_3", [0, 0.14], traps={(3, 0.14): "Three channels changed by a point or more, but none of those changes is larger than chance. Decide with the p-value, not with the size of the change."})

<details>
<summary>Hint</summary>

Print `p_values` next to `change`. How many channels have a p-value below 0.05?

</details>

<details>
<summary>Solution</summary>

```python
moved = [channel for channel in channels if p_values[channel] < 0.05]
answer_3 = [len(moved), round(min(p_values.values()), 2)]
```

Three channels changed by a point or more (app_store -1.81, organic_search -1.03 and paid_search +1.71 points), yet no change is larger than chance: the smallest p-value is 0.14, for app_store. A size cutoff ignores how many people each channel has.

</details>

## Run all from the top

Restart the session, run every cell from the first, then read this cell last.

In [ ]:
for label in EXERCISES:
    state = RESULTS.get(label)
    print(f"{label}: " + ("correct" if state else "not yet" if state is False else "not started"))
print(f"{sum(1 for label in EXERCISES if RESULTS.get(label))} of {len(EXERCISES)} exercises correct.")